In [ ]:
import ragas
print(f"✓ Ragas {ragas.__version__} imports cleanly")

In [ ]:
# Run this in a fresh kernel after `uv add pyarrow`
import pyarrow
import datasets
import sentence_transformers
import ragas
print(f"✓ pyarrow {pyarrow.__version__}")
print(f"✓ datasets {datasets.__version__}")
print(f"✓ sentence-transformers {sentence_transformers.__version__}")
print(f"✓ ragas {ragas.__version__}")

In [ ]:
import sys
from pathlib import Path

# Ensure notebooks/ is on sys.path so _setup.py can be imported
sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"))

from _setup import PROJECT_ROOT, retriever

In [11]:
"""
Smoke test: smallest possible Claude call.
"""
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=True)
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not loaded"
print(f"✓ Key loaded (starts with: {os.getenv('ANTHROPIC_API_KEY')[:10]}...)")

from anthropic import Anthropic
client = Anthropic()  # picks up ANTHROPIC_API_KEY from env

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=32,
    messages=[{"role": "user", "content": "Reply with just the word OK."}],
)
print(f"Response: {response.content[0].text}")
print(f"Usage: {response.usage.input_tokens} in, {response.usage.output_tokens} out")
print(f"Cost for this call: ${(response.usage.input_tokens * 1 + response.usage.output_tokens * 5) / 1_000_000:.6f}")

✓ Key loaded (starts with: sk-ant-api...)
Response: OK
Usage: 14 in, 4 out
Cost for this call: $0.000034


In [ ]:
"""
Pipeline test with Claude as generator.
"""
import importlib
import src.generation
importlib.reload(src.generation)
from src.generation import Generator

import src.pipeline
importlib.reload(src.pipeline)
from src.pipeline import RAGPipeline

generator = Generator(provider="anthropic", prompt_version="v2_enforced_shape")
pipeline = RAGPipeline(retriever=retriever, generator=generator, top_k=5)

response = pipeline.query("What cybersecurity risks does JPMorgan disclose?")
pipeline.pretty_print(response)

QUESTION: What cybersecurity risks does JPMorgan disclose?

ANSWER:
# JPMorgan Chase Cybersecurity Risks Disclosure

JPMorgan Chase discloses a comprehensive set of cybersecurity risks across multiple dimensions:

**Nature and Evolution of Cyber Attacks**

[Chunk 1] The bank notes that "the techniques used in cyber attacks evolve frequently and increase in sophistication, and therefore a cyber attack may not be recognized until launched or may go undetected for extended periods." [Chunk 1] Additionally, "a third-party, after establishing a foothold on an internal network without being detected, may gain access to other networks and systems," and [Chunk 1] "cyber attacks can originate from a wide variety of sources, including certain threat actors that are well-resourced and can sustain malicious activities for extended periods."

**Intensifying Factors**

[Chunk 1] JPMorgan identifies risks intensified by "increased volume and complexity of cyber attacks during periods of heightened ge

In [ ]:
from pathlib import Path

cache_path = PROJECT_ROOT / "data" / "pipeline_outputs_v2.pkl"

if cache_path.exists():
    size = cache_path.stat().st_size
    print(f"Cache file size: {size} bytes")
    
    if size < 100:  # empty or near-empty
        cache_path.unlink()
        print(f"✓ Deleted corrupt cache file")
    else:
        print(f"Cache looks valid (size > 100 bytes), not deleting")
else:
    print("No cache file exists")

In [15]:
"""
Build dataset_records with Claude. Expected runtime: 2-3 minutes total.
"""
import json
import pickle
from pathlib import Path

cache_path = PROJECT_ROOT / "data" / "pipeline_outputs_claude_v2.pkl"
cache_path.parent.mkdir(exist_ok=True)

generator = Generator(provider="anthropic", prompt_version="v2_enforced_shape")
pipeline = RAGPipeline(retriever=retriever, generator=generator, top_k=5)

test_data = json.loads((PROJECT_ROOT / "data" / "test_set"/"test_set.json").read_text())
questions_to_run = [q for q in test_data if not q["should_refuse"]]

dataset_records = []
total_input_tokens = 0
total_output_tokens = 0

for i, q in enumerate(questions_to_run, start=1):
    print(f"[{i}/{len(questions_to_run)}] {q['id']}: {q['question'][:60]}...")
    response = pipeline.query(q["question"])
    
    # Track token usage so you can verify spend
    if hasattr(response.retrieved_chunks[0], '__class__'):
        raw = generator.client  # just for clarity — actual usage on response object
    # Get usage from the raw response (it's in generation.py's raw_response)
    # We need to dig into the result — simplest: re-run the generation manually for accounting,
    # but for now just trust the cost estimate
    
    dataset_records.append({
        "question": q["question"],
        "answer": response.answer,
        "contexts": [c.text for c in response.retrieved_chunks],
        "ground_truth": q["reference_answer"],
    })

# Atomic save
tmp_path = cache_path.with_suffix(".pkl.tmp")
with open(tmp_path, "wb") as f:
    pickle.dump(dataset_records, f)
tmp_path.replace(cache_path)

print(f"\n✓ Built {len(dataset_records)} records, cached to {cache_path.name}")
print(f"Estimated cost: ~$0.10-0.20 (check console.anthropic.com → Usage for exact)")

[1/16] single_01: What cybersecurity risks does JPMorgan disclose in its Risk ...
[2/16] single_02: How does Capital One describe its commercial real estate exp...
[3/16] single_03: What climate-related risks does BlackRock identify?...
[4/16] single_04: What are PayPal's main risks from third-party service provid...
[5/16] single_05: What interest rate risks does Goldman Sachs disclose?...
[6/16] single_06: What geopolitical risks does Visa identify?...
[7/16] comparison_01: How do JPM and GS describe their stress-testing and capital ...
[8/16] comparison_02: Compare the cybersecurity risk disclosures of Mastercard and...
[9/16] comparison_03: How do BAC and COF differ in how they describe consumer cred...
[10/16] comparison_04: How do PYPL and V describe their exposure to cryptocurrency ...
[11/16] comparison_05: How do AXP and BLK describe their climate-related risks?...
[12/16] specific_01: Which filings discuss CCAR stress testing requirements?...
[13/16] specific_02: Which compan

In [5]:
"""
Stub the missing module so Ragas's eager import succeeds.
We don't use Vertex AI, so creating an empty placeholder is safe.
RUN THIS BEFORE 'from ragas import evaluate'.
"""
import sys
import types

# Create a fake module hierarchy that Ragas's import expects
if "langchain_community.chat_models.vertexai" not in sys.modules:
    fake_vertexai = types.ModuleType("langchain_community.chat_models.vertexai")
    # Define a placeholder class so the import succeeds
    class ChatVertexAI:
        def __init__(self, *args, **kwargs):
            raise NotImplementedError("Vertex AI is not configured in this project")
    fake_vertexai.ChatVertexAI = ChatVertexAI
    sys.modules["langchain_community.chat_models.vertexai"] = fake_vertexai

# Also check the second import — ragas imports VertexAI from langchain_community.llms too
try:
    from langchain_community.llms import VertexAI  # noqa
except ImportError:
    import langchain_community.llms
    class VertexAI:
        def __init__(self, *args, **kwargs):
            raise NotImplementedError("Vertex AI is not configured in this project")
    langchain_community.llms.VertexAI = VertexAI

print("✓ Vertex AI stub created — Ragas imports should now succeed")

✓ Vertex AI stub created — Ragas imports should now succeed


In [8]:
"""
Ragas evaluation with Claude Haiku as the judge model.

Architecture:
- Judge LLM: Claude Haiku 4.5 (paid, fast, no rate limit drama)
- Embeddings: local BGE (same model as the retriever — no API needed)
- Concurrency: max_workers=4 (Claude's rate limits comfortably accommodate this)
"""
import pickle
from datasets import Dataset


from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

from langchain_anthropic import ChatAnthropic
from langchain_huggingface import HuggingFaceEmbeddings

# ----------------------------------------------------------------------
# Load the cached pipeline outputs (from Step 6 of your Claude setup)
# ----------------------------------------------------------------------
cache_path = PROJECT_ROOT / "data" / "pipeline_outputs_claude_v2.pkl"
with open(cache_path, "rb") as f:
    dataset_records = pickle.load(f)
print(f"✓ Loaded {len(dataset_records)} records from cache")

# ----------------------------------------------------------------------
# Configure the judge model (Claude Haiku 4.5)
# ----------------------------------------------------------------------
judge_llm = LangchainLLMWrapper(
    ChatAnthropic(
        model="claude-haiku-4-5-20251001",
        temperature=0,
        max_tokens=4096,
    )
)

# ----------------------------------------------------------------------
# Configure embeddings — reuse your local BGE model
# Same model name your retriever uses, so it loads from cache (no re-download)
# ----------------------------------------------------------------------
judge_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
)

# ----------------------------------------------------------------------
# Run config — much less paranoid than the Gemini version
# Claude has higher rate limits, so we can parallelize a bit
# ----------------------------------------------------------------------
run_config = RunConfig(
    timeout=180,         # individual call timeout (sec)
    max_retries=5,       # retries per failed call
    max_wait=30,         # max backoff (sec)
    max_workers=4,       # 4 parallel calls — Claude can handle it easily
)

# ----------------------------------------------------------------------
# Configure metrics — attach judge LLM and embeddings to each
# ----------------------------------------------------------------------
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
for m in metrics:
    m.llm = judge_llm
    if hasattr(m, "embeddings"):
        m.embeddings = judge_embeddings

# ----------------------------------------------------------------------
# Build dataset and run evaluation
# ----------------------------------------------------------------------
dataset = Dataset.from_list(dataset_records)

print("Starting Ragas evaluation — expect 3-6 minutes with Claude")
results = evaluate(
    dataset=dataset,
    metrics=metrics,
    run_config=run_config,
    raise_exceptions=False,
)

print("\n=== Ragas Aggregate Results ===")
print(results)

✓ Loaded 16 records from cache


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Starting Ragas evaluation — expect 3-6 minutes with Claude


Evaluating:   0%|          | 0/64 [00:00<?, ?it/s]


=== Ragas Aggregate Results ===
{'faithfulness': 0.6816, 'answer_relevancy': 0.7115, 'context_precision': 0.2937, 'context_recall': 0.1927}


In [ ]:
import numpy as np

df = results.to_pandas()

print("=" * 60)
print("Per-metric success rate")
print("=" * 60)
for col in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    if col in df.columns:
        valid = df[col].notna().sum()
        total = len(df)
        if valid > 0:
            avg = df[col].dropna().mean()
            print(f"  {col:<22} {valid}/{total} valid | avg score: {avg:.3f}")
        else:
            print(f"  {col:<22} 0/{total} valid | NO SCORES")
print()

# Show rows with failures so we know which questions to look at
print("Rows with any NaN (failed judgments):")
failed = df[df.isna().any(axis=1)]
if len(failed) > 0:
    print(failed[["user_input"]].to_string() if "user_input" in failed.columns 
          else failed[["question"]].to_string())

Per-metric success rate
  faithfulness           16/16 valid | avg score: 0.682
  answer_relevancy       16/16 valid | avg score: 0.712
  context_precision      16/16 valid | avg score: 0.294
  context_recall         16/16 valid | avg score: 0.193

Rows with any NaN (failed judgments):


: 

log1
============================================================
Per-metric success rate
============================================================
  faithfulness           16/16 valid | avg score: 0.682 --Below typical. Means about 32% of claims in your answers don't trace cleanly to retrieved chunks.
  answer_relevancy       16/16 valid | avg score: 0.712 --Reasonable. Answers are on-topic for the questions asked.
  context_precision      16/16 valid | avg score: 0.294 --Bad. Most retrieved chunks aren't actually useful for the answer.
  context_recall         16/16 valid | avg score: 0.193 --Bad. Top-5 chunks frequently miss info needed to answer.

Rows with any NaN (failed judgments):

Hypothesis A: Retrieval is actually bad. 
    The RRF combination of dense + BM25 may not be tuned right. 
    top_k=5 might be too small (info gets buried at rank 6+)
    Chunks themselves might be too long, diluting relevance signal
Hypothesis B: Reference answer over-specify.
    Retrieval might actually be fine.
     reference answers need to be rewritten.
     The 0.19 context_recall is measuring "ground truth includes claims that aren't in the corpus at all," not "retrieval missed important chunks"

In [ ]:
from pathlib import Path

candidate_paths = [
    PROJECT_ROOT / "data" / "ragas_run_v2_baseline.csv",
    PROJECT_ROOT / "data" / "ragas_run_baseline.csv",
]

for p in candidate_paths:
    if p.exists():
        print(f"✓ Found: {p}  ({p.stat().st_size:,} bytes)")
    else:
        print(f"✗ Not found: {p.name}")

In [ ]:
"""
Diagnose why context_precision and context_recall are low.

For 3 sample questions, show:
- The question
- Your reference answer (ground truth)
- The system's actual answer  
- The chunks that were retrieved
- The Ragas score for each metric on this question

Then YOU decide: is the chunk relevant to the reference? Did the chunks
contain enough to construct the reference?
"""
import pandas as pd

# Make pandas show full text
pd.set_option("display.max_colwidth", None)

df = results.to_pandas()
print("DataFrame columns:", df.columns.tolist())
print()

# Pick 3 questions to look at — one each from different categories if possible
# Print just the metric columns for orientation
score_cols = [c for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"] if c in df.columns]
print("All questions and their scores:\n")
print(df[["user_input"] + score_cols].to_string() if "user_input" in df.columns 
      else df[["question"] + score_cols].to_string())